# Independent — XGBoost + LightGBM + LogReg per target

Demonstrates `IndependentMultiTargetEstimator`: one scikit-rec sub-estimator per target, with mixed sub-estimator types in the same model. Multilabel groups fan out into per-member binary classifiers.

**No torch in this notebook** — that's covered in [joint_families.ipynb](joint_families.ipynb). On macOS, mixing torch and lightgbm in the same kernel triggers an OpenBLAS+Accelerate collision; splitting by backend avoids it.

Independent + conditional is NOT supported (v3 locked decision #1). Use a joint family for conditional inference; see [conditional.ipynb](conditional.ipynb).

In [1]:
# ruff: noqa: E402  (thread-pool env vars must be set before imports)
# macOS BLAS/OpenMP guard — set BEFORE numpy/lightgbm/xgboost import.
import os

for _var in ("OMP_NUM_THREADS", "MKL_NUM_THREADS", "VECLIB_MAXIMUM_THREADS", "OPENBLAS_NUM_THREADS"):
    os.environ.setdefault(_var, "1")

# Pre-import all tree-backend libraries at the TOP, before anything else
# touches numpy. On macOS, importing lightgbm and xgboost in different
# orders relative to sklearn (which imports numpy) can trigger an OpenBLAS+
# Accelerate kernel-death later in the notebook. Explicit top-of-file
# imports pin the initialization order.
import warnings

import lightgbm  # noqa: F401, E402
import sklearn  # noqa: F401, E402
import xgboost  # noqa: F401, E402

warnings.filterwarnings("ignore", category=UserWarning)

import numpy as np
import pandas as pd

from skrec.constants import USER_ID_NAME
from skrec.estimator.classification import IndependentMultiTargetEstimator
from skrec.estimator.classification.lightgbm_classifier import LightGBMClassifierEstimator
from skrec.estimator.classification.logreg_classifier import LogisticRegressionClassifierEstimator
from skrec.estimator.classification.xgb_classifier import XGBClassifierEstimator
from skrec.estimator.regression.lightgbm_regressor import LightGBMRegressorEstimator
from skrec.evaluator.datatypes import RecommenderEvaluatorType
from skrec.metrics.datatypes import RecommenderMetricType
from skrec.orchestrator import (
    TargetGroupSpec,
    TargetType,
    capability_matrix,
    create_recommender_pipeline,
)
from skrec.recommender.ranking.ranking_recommender import RankingRecommender
from skrec.scorer.mixed_type_multi_target import MixedTypeMultiTargetScorer

print("Per-target sub-estimator compatibility (from capability_matrix):")
for tt, allowed in capability_matrix()["independent_target_compat"].items():
    print(f"  {tt:12s} → {allowed}")

Per-target sub-estimator compatibility (from capability_matrix):
  binary       → ('lightgbm', 'logreg', 'sklearn', 'xgboost')
  regression   → ('lightgbm', 'sklearn', 'xgboost')
  multiclass   → ('lightgbm', 'logreg')
  multilabel   → ('lightgbm', 'logreg', 'sklearn', 'xgboost')


## 1. Synthetic data — same shape as the joint-families notebook

5 features, 800 users, four targets of different types. Built identically so the leaderboard at the bottom is directly comparable to the joint-families notebook's.

In [2]:
def make_synthetic(n=800, seed=42):
    rng = np.random.default_rng(seed)
    X = pd.DataFrame(rng.normal(size=(n, 5)), columns=[f"feat_{i}" for i in range(5)])
    y_binary = (X["feat_0"] > 0).astype(int).to_numpy()
    y_reg = (2.5 * X["feat_1"] + rng.normal(scale=0.2, size=n)).to_numpy()
    cls_idx = np.column_stack([X["feat_2"], X["feat_3"], X["feat_4"]]).argmax(axis=1)
    y_mc = np.array(["action_A", "action_B", "action_C"])[cls_idx]
    y_ml = np.column_stack(
        [
            (X["feat_2"] > 0).astype(int).to_numpy(),
            (X["feat_3"] > 0).astype(int).to_numpy(),
        ]
    )
    target_specs = {
        "ITEM_clicked": TargetType.BINARY,
        "ITEM_revenue": TargetType.REGRESSION,
        "ITEM_action": TargetType.MULTICLASS,
        "engagement": TargetGroupSpec(
            type=TargetType.MULTILABEL,
            columns=["ITEM_email_open", "ITEM_app_open"],
        ),
    }
    y = {
        "ITEM_clicked": y_binary,
        "ITEM_revenue": y_reg,
        "ITEM_action": y_mc,
        "engagement": y_ml,
    }
    return X, y, target_specs


X, y, target_specs = make_synthetic(n=800)
split = 600
X_train, X_valid = X.iloc[:split], X.iloc[split:].reset_index(drop=True)
y_train = {k: v[:split] for k, v in y.items()}
y_valid = {k: v[split:] for k, v in y.items()}
print("train:", X_train.shape, "  valid:", X_valid.shape)

train: (600, 5)   valid: (200, 5)


## 2. Direct construction — mix XGB, LightGBM, LogReg by target

Independent mode's headline feature: pick the right sub-estimator per target. XGBoost for the binary target, LightGBM for regression and multiclass, LogReg for one multilabel member.

**Multiclass note**: XGBClassifierEstimator's `inplace_predict` returns malformed shapes on multiclass; `independent_target_compat` excludes it from the allowed list. The scorer-side defensive guard catches the misconfiguration cleanly at predict time if a caller bypasses the compat table.

In [3]:
independent = IndependentMultiTargetEstimator(
    target_specs=target_specs,
    estimators={
        "ITEM_clicked": XGBClassifierEstimator(params={"n_estimators": 50, "max_depth": 3}),
        "ITEM_revenue": LightGBMRegressorEstimator(params={"n_estimators": 100, "verbose": -1}),
        "ITEM_action": LightGBMClassifierEstimator(params={"n_estimators": 50, "verbose": -1}),
        "ITEM_email_open": XGBClassifierEstimator(params={"n_estimators": 50, "max_depth": 3}),
        "ITEM_app_open": LogisticRegressionClassifierEstimator(params={"max_iter": 200}),
    },
)
independent.fit(X_train, y_train)
print("independent trained; per-target sub-estimators:")
for k, v in independent.estimators.items():
    print(f"  {k:20s} → {type(v).__name__}")

independent trained; per-target sub-estimators:
  ITEM_clicked         → XGBClassifierEstimator
  ITEM_revenue         → LightGBMRegressorEstimator
  ITEM_action          → LightGBMClassifierEstimator
  ITEM_email_open      → XGBClassifierEstimator
  ITEM_app_open        → LogisticRegressionClassifierEstimator


## 3. Per-target evaluation — `Dict[str, float]`

Same evaluation contract as the joint families. Each `TargetType` gets its appropriate metric; the result is always a per-target dict.

In [4]:
scorer = MixedTypeMultiTargetScorer(estimator=independent, target_specs=target_specs)
recommender = RankingRecommender(scorer=scorer)

valid_inf = X_valid.copy()
valid_inf.insert(0, USER_ID_NAME, [f"u_{i}" for i in range(len(X_valid))])

logged = pd.DataFrame(
    {
        "ITEM_clicked": y_valid["ITEM_clicked"],
        "ITEM_revenue": y_valid["ITEM_revenue"],
        "ITEM_action": y_valid["ITEM_action"],
        "ITEM_email_open": y_valid["engagement"][:, 0],
        "ITEM_app_open": y_valid["engagement"][:, 1],
    }
)

result = recommender.evaluate(
    eval_type=RecommenderEvaluatorType.SIMPLE,
    metric_type={
        "ITEM_clicked": RecommenderMetricType.ROC_AUC,
        "ITEM_revenue": RecommenderMetricType.MAE,
        "ITEM_action": RecommenderMetricType.MULTICLASS_ACCURACY,
        "ITEM_email_open": RecommenderMetricType.ROC_AUC,
        "ITEM_app_open": RecommenderMetricType.ROC_AUC,
    },
    eval_top_k=10,
    score_items_kwargs={"interactions": valid_inf},
    eval_kwargs={"logged_rewards": logged},
)
print("Per-target metrics:")
for k, v in result.items():
    print(f"  {k:20s} = {v:.4f}")

Per-target metrics:
  ITEM_clicked         = 0.9951
  ITEM_revenue         = 0.2185
  ITEM_action          = 0.9600
  ITEM_email_open      = 1.0000
  ITEM_app_open        = 1.0000


## 4. Factory-driven construction (the orchestrator path)

What scikit-rec-agent and config-driven callers use. Sub-estimators are composed from `defaults` (one per target type) with optional `per_target` overrides for specific columns. Multilabel members consult the `multilabel` default key by default.

In [5]:
config = {
    "recommender_type": "ranking",
    "scorer_type": "mixed_type_multi_target",
    "scorer_config": {"target_specs": target_specs},
    "estimator_config": {
        "estimator_type": "tabular",
        "ml_task": "multi_target",
        "multi_target": {
            "mode": "independent",
            "independent": {
                "defaults": {
                    "binary": {"estimator_type": "xgboost", "params": {"n_estimators": 30}},
                    "regression": {"estimator_type": "lightgbm", "params": {"n_estimators": 30, "verbose": -1}},
                    "multiclass": {"estimator_type": "lightgbm", "params": {"n_estimators": 30, "verbose": -1}},
                    "multilabel": {"estimator_type": "xgboost", "params": {"n_estimators": 30}},
                },
                "per_target": {
                    # Member-level overrides take precedence over the 'multilabel' default.
                    "ITEM_app_open": {"estimator_type": "logreg", "params": {"max_iter": 200}},
                },
            },
        },
    },
}
factory_recommender = create_recommender_pipeline(config)
print("Factory-built recommender:", type(factory_recommender).__name__)
print("  scorer  :", type(factory_recommender.scorer).__name__)
print("  estimator:", type(factory_recommender.scorer.estimator).__name__)
print("  per-target estimators:")
for k, v in factory_recommender.scorer.estimator.estimators.items():
    print(f"    {k:20s} → {type(v).__name__}")

2026-05-26 04:49:27,481 - skrec.orchestrator.factory - INFO Creating recommender pipeline from config...


2026-05-26 04:49:27,482 - skrec.orchestrator.factory - INFO Creating estimator. Estimator type: tabular


2026-05-26 04:49:27,482 - skrec.orchestrator.factory - INFO Creating IndependentMultiTargetEstimator with 5 sub-estimators


2026-05-26 04:49:27,482 - skrec.orchestrator.factory - INFO Creating scorer of type: mixed_type_multi_target


2026-05-26 04:49:27,482 - skrec.orchestrator.factory - INFO Creating recommender of type: ranking


2026-05-26 04:49:27,482 - skrec.orchestrator.factory - INFO Recommender pipeline created successfully.


Factory-built recommender: RankingRecommender
  scorer  : MixedTypeMultiTargetScorer
  estimator: IndependentMultiTargetEstimator
  per-target estimators:
    ITEM_clicked         → XGBClassifierEstimator
    ITEM_revenue         → LightGBMRegressorEstimator
    ITEM_action          → LightGBMClassifierEstimator
    ITEM_email_open      → XGBClassifierEstimator
    ITEM_app_open        → LogisticRegressionClassifierEstimator


## 5. Vanilla rejects `OBSERVED_*` at inference

The independent family is **vanilla** — `OBSERVED_*` columns at inference are rejected with a pointer to the conditional joint families. See [conditional.ipynb](conditional.ipynb) for the v3 conditional path (joint MLP / joint Transformer only).

In [6]:
bad_inf = valid_inf.head(5).copy()
bad_inf["OBSERVED_clicked"] = 1
try:
    scorer.score_items(interactions=bad_inf)
except NotImplementedError as e:
    print("Vanilla independent rejects OBSERVED_*:")
    print(" ", str(e).split(".")[0])

Vanilla independent rejects OBSERVED_*:
  OBSERVED_* columns require a ConditionalMultiTargetEstimator (e


## 6. HPO round-trip — sweep `per_target` hyperparameters

The plan's HPO contract for `mode="independent"` uses flat dotted-path keys (`multi_target.independent.per_target.<name>.params.<param>`). Below is a minimal sweep: vary `n_estimators` for `ITEM_revenue` (the LightGBM regressor) and pick the config that minimizes MAE on the valid slice. End-to-end through the factory each iteration.

In [7]:
# Walk a small grid of n_estimators values for ITEM_revenue's sub-estimator.
# Each iteration: build a fresh recommender via create_recommender_pipeline,
# train on the train split, evaluate per-target metrics on the valid split,
# record the ITEM_revenue MAE.
sweep_results = []
for n_estimators in [10, 50, 100, 200]:
    cfg = {
        "recommender_type": "ranking",
        "scorer_type": "mixed_type_multi_target",
        "scorer_config": {"target_specs": target_specs},
        "estimator_config": {
            "estimator_type": "tabular",
            "ml_task": "multi_target",
            "multi_target": {
                "mode": "independent",
                "independent": {
                    "defaults": {
                        "binary": {"estimator_type": "xgboost", "params": {"n_estimators": 30}},
                        "regression": {"estimator_type": "lightgbm", "params": {"n_estimators": 30, "verbose": -1}},
                        "multiclass": {"estimator_type": "lightgbm", "params": {"n_estimators": 30, "verbose": -1}},
                        "multilabel": {"estimator_type": "xgboost", "params": {"n_estimators": 30}},
                    },
                    "per_target": {
                        # The sweep knob.
                        "ITEM_revenue": {
                            "estimator_type": "lightgbm",
                            "params": {"n_estimators": n_estimators, "verbose": -1},
                        },
                    },
                },
            },
        },
    }
    rec_hpo = create_recommender_pipeline(cfg)
    # Manual train: process_datasets → train_model on the train split.
    train_inf = X_train.copy()
    train_inf.insert(0, USER_ID_NAME, [f"t{i}" for i in range(len(X_train))])
    for k in ("ITEM_clicked", "ITEM_revenue", "ITEM_action"):
        train_inf[k] = y_train[k]
    train_inf["ITEM_email_open"] = y_train["engagement"][:, 0]
    train_inf["ITEM_app_open"] = y_train["engagement"][:, 1]
    X_train_proc, y_train_dict = rec_hpo.scorer.process_datasets(
        interactions_df=train_inf,
        is_training=True,
    )
    rec_hpo.scorer.train_model(X_train_proc, y_train_dict)

    out = rec_hpo.evaluate(
        eval_type=RecommenderEvaluatorType.SIMPLE,
        metric_type={
            "ITEM_revenue": RecommenderMetricType.MAE,
            "ITEM_clicked": RecommenderMetricType.ROC_AUC,
            "ITEM_action": RecommenderMetricType.MULTICLASS_ACCURACY,
            "ITEM_email_open": RecommenderMetricType.ROC_AUC,
            "ITEM_app_open": RecommenderMetricType.ROC_AUC,
        },
        eval_top_k=10,
        score_items_kwargs={"interactions": valid_inf},
        eval_kwargs={"logged_rewards": logged},
    )
    sweep_results.append(
        {
            "n_estimators": n_estimators,
            "ITEM_revenue_MAE": out["ITEM_revenue"],
            "ITEM_clicked_AUC": out["ITEM_clicked"],
        }
    )

sweep_df = pd.DataFrame(sweep_results).set_index("n_estimators").round(4)
print("HPO sweep over ITEM_revenue.params.n_estimators:")
print(sweep_df)
best = sweep_df["ITEM_revenue_MAE"].idxmin()
print(f"\nBest n_estimators (lowest ITEM_revenue MAE): {best}")

2026-05-26 04:49:27,490 - skrec.orchestrator.factory - INFO Creating recommender pipeline from config...


2026-05-26 04:49:27,491 - skrec.orchestrator.factory - INFO Creating estimator. Estimator type: tabular


2026-05-26 04:49:27,491 - skrec.orchestrator.factory - INFO Creating IndependentMultiTargetEstimator with 5 sub-estimators


2026-05-26 04:49:27,491 - skrec.orchestrator.factory - INFO Creating scorer of type: mixed_type_multi_target


2026-05-26 04:49:27,491 - skrec.orchestrator.factory - INFO Creating recommender of type: ranking


...

2026-05-26 04:49:29,298 - skrec.orchestrator.factory - INFO Creating IndependentMultiTargetEstimator with 5 sub-estimators


2026-05-26 04:49:29,298 - skrec.orchestrator.factory - INFO Creating scorer of type: mixed_type_multi_target


2026-05-26 04:49:29,298 - skrec.orchestrator.factory - INFO Creating recommender of type: ranking


2026-05-26 04:49:29,298 - skrec.orchestrator.factory - INFO Recommender pipeline created successfully.


HPO sweep over ITEM_revenue.params.n_estimators:
              ITEM_revenue_MAE  ITEM_clicked_AUC
n_estimators                                    
10                      0.7479            0.9951
50                      0.2095            0.9951
100                     0.2185            0.9951
200                     0.2263            0.9951

Best n_estimators (lowest ITEM_revenue MAE): 50


## Where to go next

- **Joint families (PyTorch)**: [joint_families.ipynb](joint_families.ipynb) — joint MLP + joint Transformer
- **Conditional inference (v3)**: [conditional.ipynb](conditional.ipynb) — real-time-label conditioning (joint families only)
- **Decision rule**: [docs/user-guide/decision-rule.md](../../docs/user-guide/decision-rule.md) — joint vs independent
- **Scorer reference**: [docs/user-guide/scorers.md](../../docs/user-guide/scorers.md#5-mixedtypemultitargetscorer)